In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate, KFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

In [2]:
train_sample_path = "../train_sample.csv"
test_sample_path = "../test_sample.csv"

In [3]:
train_sample = pd.read_csv(train_sample_path)
train_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   start_point                    40000 non-null  str    
 1   end_point                      40000 non-null  str    
 2   time_of_day                    40000 non-null  str    
 3   day_of_week                    40000 non-null  str    
 4   traffic_condition              25599 non-null  float64
 5   event_count                    40000 non-null  int64  
 6   is_holiday                     40000 non-null  int64  
 7   vehicle_density                25622 non-null  str    
 8   population_density             25552 non-null  str    
 9   weather                        25571 non-null  str    
 10  public_transport_availability  40000 non-null  int64  
 11  historical_delay_factor        40000 non-null  float64
 12  travel_time                    40000 non-null  float64
dt

In [4]:
test_sample = pd.read_csv(test_sample_path)
test_sample.head(2)

,start_point,end_point,time_of_day,day_of_week,traffic_condition,event_count,is_holiday,vehicle_density,population_density,weather,public_transport_availability,historical_delay_factor
0,West Jakarta (Jakarta Barat),East Jakarta (Jakarta Timur),morning,Saturday,5.0,8,1,medium,NaN,NaN,2,1.126429
1,South Jakarta (Jakarta Selatan),East Jakarta (Jakarta Timur),evening,Saturday,NaN,9,1,low,medium,fog,2,1.121015


In [5]:
train_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   start_point                    40000 non-null  str    
 1   end_point                      40000 non-null  str    
 2   time_of_day                    40000 non-null  str    
 3   day_of_week                    40000 non-null  str    
 4   traffic_condition              25599 non-null  float64
 5   event_count                    40000 non-null  int64  
 6   is_holiday                     40000 non-null  int64  
 7   vehicle_density                25622 non-null  str    
 8   population_density             25552 non-null  str    
 9   weather                        25571 non-null  str    
 10  public_transport_availability  40000 non-null  int64  
 11  historical_delay_factor        40000 non-null  float64
 12  travel_time                    40000 non-null  float64
dt

In [6]:
start = train_sample["start_point"].str.split().str[0]
end = train_sample["end_point"].str.split().str[0]

train_sample["start_end_point"] = start + " " + end
test_sample["start_end_point"] = test_sample["start_point"].str.split().str[0] + " " + test_sample["end_point"].str.split().str[0]

In [7]:
train_sample['start_end_point'].value_counts()

start_end_point
Central West     4153
West South       4048
Central South    4016
South East       4013
North West       4010
Central East     3987
Central North    3964
North East       3948
North South      3935
West East        3926
Name: count, dtype: int64

Based on EDA on missing_values_handling.ipynb.

if start_point == 'Central' then vehicle_density = 'high'


else vehicle_density = 'medium'

In [8]:
train_sample["vehicle_density"] = (
    train_sample["vehicle_density"]
    .fillna(
        train_sample["start_end_point"]
        .str.split()
        .str[0]
        .eq("Central")
        .map({True: "high", False: "medium"})
    )
)

In [9]:
test_sample["vehicle_density"] = (
    test_sample["vehicle_density"]
    .fillna(
        test_sample["start_end_point"]
        .str.split()
        .str[0]
        .eq("Central")
        .map({True: "high", False: "medium"})
    )
)

In [10]:
# Numeric → mean
numeric_cols = train_sample.select_dtypes(include='number').columns

train_sample[numeric_cols] = train_sample[numeric_cols].fillna(
    train_sample[numeric_cols].mean()
)

# Categorical → mode
categorical_cols = train_sample.select_dtypes(
    include=['str']
).columns

for col in categorical_cols:
    train_sample[col] = train_sample[col].fillna(
        train_sample[col].mode()[0]
    )

train_sample.isnull().sum()

start_point                      0
end_point                        0
time_of_day                      0
day_of_week                      0
traffic_condition                0
event_count                      0
is_holiday                       0
vehicle_density                  0
population_density               0
weather                          0
public_transport_availability    0
historical_delay_factor          0
travel_time                      0
start_end_point                  0
dtype: int64

In [13]:
# Numeric → mean
numeric_cols = test_sample.select_dtypes(include='number').columns

test_sample[numeric_cols] = test_sample[numeric_cols].fillna(
    test_sample[numeric_cols].mean()
)

# Categorical → mode
categorical_cols = test_sample.select_dtypes(
    include=['str']
).columns

for col in categorical_cols:
    test_sample[col] = test_sample[col].fillna(
        test_sample[col].mode()[0]
    )

test_sample.isnull().sum()
test_sample.isnull().sum()

start_point                      0
end_point                        0
time_of_day                      0
day_of_week                      0
traffic_condition                0
event_count                      0
is_holiday                       0
vehicle_density                  0
population_density               0
weather                          0
public_transport_availability    0
historical_delay_factor          0
start_end_point                  0
dtype: int64

In [11]:
train_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   start_point                    40000 non-null  str    
 1   end_point                      40000 non-null  str    
 2   time_of_day                    40000 non-null  str    
 3   day_of_week                    40000 non-null  str    
 4   traffic_condition              40000 non-null  float64
 5   event_count                    40000 non-null  int64  
 6   is_holiday                     40000 non-null  int64  
 7   vehicle_density                40000 non-null  str    
 8   population_density             40000 non-null  str    
 9   weather                        40000 non-null  str    
 10  public_transport_availability  40000 non-null  int64  
 11  historical_delay_factor        40000 non-null  float64
 12  travel_time                    40000 non-null  float64
 1

In [14]:
test_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   start_point                    3000 non-null   str    
 1   end_point                      3000 non-null   str    
 2   time_of_day                    3000 non-null   str    
 3   day_of_week                    3000 non-null   str    
 4   traffic_condition              3000 non-null   float64
 5   event_count                    3000 non-null   int64  
 6   is_holiday                     3000 non-null   int64  
 7   vehicle_density                3000 non-null   str    
 8   population_density             3000 non-null   str    
 9   weather                        3000 non-null   str    
 10  public_transport_availability  3000 non-null   int64  
 11  historical_delay_factor        3000 non-null   float64
 12  start_end_point                3000 non-null   object 
dtyp

train val split

In [15]:
X_train, X_val, y_train, y_val = train_test_split(train_sample.drop(columns=['travel_time']), train_sample['travel_time'], test_size=0.2, random_state=38)

In [16]:
X_test = test_sample

In [17]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_train index:", X_train.index[:5])
print("y_train index:", y_train.index[:5])

X_train: (32000, 13)
y_train: (32000,)
X_train index: Index([10777, 35199, 38951, 3007, 31110], dtype='int64')
y_train index: Index([10777, 35199, 38951, 3007, 31110], dtype='int64')


one-hot encoding

In [14]:
# ohe = OneHotEncoder(sparse=False, drop='first', handle_unknown='ignore')
# X_train_encoded = ohe.fit_transform(X_train)
# X_val_encoded = ohe.transform(X_val)
# X_test_encoded = ohe.transform(test_sample)
# X_train_encoded.head(3)

In [18]:
categorical_cols = train_sample.select_dtypes(include=['str', 'object']).columns

In [19]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols),
        ('num', StandardScaler(), numeric_cols)
    ]
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error']
)

print('R²:', scores['test_r2'].mean())
print('MAE:', -scores['test_neg_mean_absolute_error'].mean())
print('MSE:', -scores['test_neg_mean_squared_error'].mean())

R²: 0.8842407799188694
MAE: 3.3548933589901977
MSE: 26.23150947505291


In [20]:
model.fit(X_train, y_train)

y_pred = model.predict(test_sample)

pd.DataFrame(y_pred).to_csv('submission.csv', index=False)

In [18]:
pd.read_csv('submission.csv')['0']

0       45.077384
1       21.868653
2       23.554827
3       19.707035
4       35.426137
          ...    
2995    41.704546
2996    24.584590
2997    15.453972
2998    13.680179
2999    15.250530
Name: 0, Length: 3000, dtype: float64